# Redrob Candidate Ranker - Sandbox Demo

This notebook is a lightweight Google Colab sandbox for the AI-Powered Candidate Ranking System. It accepts a small `candidates.jsonl` sample or uses embedded sample rows, runs a CPU-only hybrid ranking pipeline, and exports a ranked CSV.


## 1. Setup
Run this cell first. No GPU is required.


In [ ]:
import json, math, re, os, textwrap
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
print('CPU-only demo ready')


CPU-only demo ready


## 2. Candidate input
Upload `candidates.jsonl` when prompted, or skip upload to use the embedded sample.


In [ ]:
USE_UPLOAD = False  # change to True in Colab to upload a candidates.jsonl sample
candidates_path = Path('candidates_sample.jsonl')

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    candidates_path = Path(next(iter(uploaded.keys())))
else:
    sample = [
        {'candidate_id':'CAND_DEMO_001','headline':'Senior Machine Learning Engineer','summary':'Built semantic search, RAG and ranking systems for product search.','skills':['Python','Elasticsearch','FAISS','Embeddings','Learning to Rank'],'years_experience':7.2,'location':'Pune','redrob_signals':{'recruiter_response_rate':0.86,'open_to_work_flag':True,'notice_period_days':30,'profile_completeness':0.92}},
        {'candidate_id':'CAND_DEMO_002','headline':'Backend Engineer','summary':'API development and SQL. Limited ML experience.','skills':['Python','FastAPI','PostgreSQL'],'years_experience':5.5,'location':'Noida','redrob_signals':{'recruiter_response_rate':0.62,'open_to_work_flag':True,'notice_period_days':60,'profile_completeness':0.78}},
        {'candidate_id':'CAND_DEMO_003','headline':'NLP Engineer','summary':'Production NLP pipelines, retrieval augmented generation and vector databases.','skills':['LangChain','Qdrant','RAG','Embeddings','NLP'],'years_experience':6.4,'location':'Hyderabad','redrob_signals':{'recruiter_response_rate':0.81,'open_to_work_flag':False,'notice_period_days':45,'profile_completeness':0.88}},
        {'candidate_id':'CAND_DEMO_004','headline':'AI Research Intern','summary':'Academic projects only; no production ML deployment.','skills':['Pytorch','Computer Vision','Research'],'years_experience':0.7,'location':'Mumbai','redrob_signals':{'recruiter_response_rate':0.40,'open_to_work_flag':True,'notice_period_days':0,'profile_completeness':0.55}},
        {'candidate_id':'CAND_DEMO_005','headline':'Search Engineer','summary':'Owned BM25, vector search, Elasticsearch relevance tuning and offline NDCG evaluation.','skills':['BM25','Elasticsearch','Vector Search','NDCG','Python'],'years_experience':6.8,'location':'Noida','redrob_signals':{'recruiter_response_rate':0.90,'open_to_work_flag':True,'notice_period_days':30,'profile_completeness':0.95}},
    ]
    with candidates_path.open('w', encoding='utf-8') as f:
        for row in sample:
            f.write(json.dumps(row) + '\n')

print('Input file:', candidates_path)


Input file: candidates_sample.jsonl


## 3. Hybrid ranker
The demo mirrors the submitted solution: local TF-IDF relevance, JD-specific rules, behavioral modifiers, and lightweight anomaly penalties.


In [ ]:
JD_QUERY = '''Senior AI or ML engineer for ranking and retrieval systems.
Must understand semantic search, embeddings, vector databases, RAG, BM25, Elasticsearch, learning to rank, evaluation metrics, and production ML systems.
Preferred locations: Pune, Noida, Hyderabad, Mumbai, Delhi NCR. 5 to 9 years experience preferred.'''

def as_text(c):
    parts = [str(c.get('headline','')), str(c.get('summary',''))]
    skills = c.get('skills', [])
    if isinstance(skills, list): parts.append(' '.join(map(str, skills)))
    else: parts.append(str(skills))
    for job in c.get('career_history', []) or []:
        parts.extend([str(job.get('title','')), str(job.get('description','')), str(job.get('company',''))])
    return ' '.join(parts)

def rule_score(c):
    text = as_text(c).lower()
    score = 0.0
    positives = ['embedding','vector','rag','retrieval','ranking','rank','bm25','elasticsearch','faiss','qdrant','milvus','opensearch','learning to rank','semantic search']
    score += min(0.45, 0.035 * sum(p in text for p in positives))
    title = str(c.get('headline','')).lower()
    if any(t in title for t in ['machine learning','ml engineer','ai engineer','nlp','search engineer','recommendation']): score += 0.20
    years = float(c.get('years_experience', c.get('experience_years', 0)) or 0)
    if 5 <= years <= 9: score += 0.18
    elif 3 <= years < 5 or 9 < years <= 12: score += 0.08
    loc = str(c.get('location','')).lower()
    if any(x in loc for x in ['pune','noida']): score += 0.10
    elif any(x in loc for x in ['hyderabad','mumbai','delhi','gurgaon']): score += 0.06
    if 'research' in title and not any(x in text for x in ['production','deployed','search','ranking','retrieval']): score -= 0.12
    return max(0, min(1, score))

def behavior(c):
    s = c.get('redrob_signals', {}) or {}
    rr = float(s.get('recruiter_response_rate', 0.65) or 0.65)
    comp = float(s.get('profile_completeness', 0.75) or 0.75)
    notice = float(s.get('notice_period_days', 60) or 60)
    open_flag = 1.05 if s.get('open_to_work_flag', False) else 1.0
    notice_mod = 1.05 if notice <= 45 else 0.95 if notice >= 90 else 1.0
    return max(0.75, min(1.15, (0.70 + 0.25*rr + 0.10*comp) * open_flag * notice_mod))

def anomaly_penalty(c):
    years = float(c.get('years_experience', c.get('experience_years', 0)) or 0)
    skills = c.get('skills', []) or []
    if years < 1 and len(skills) > 10: return 0.05
    if years < 2 and any('expert' in str(s).lower() for s in skills): return 0.30
    return 1.0

candidates = [json.loads(line) for line in candidates_path.read_text(encoding='utf-8').splitlines() if line.strip()]
texts = [as_text(c) for c in candidates]
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, stop_words='english')
X = vectorizer.fit_transform(texts + [JD_QUERY])
text_scores = cosine_similarity(X[:-1], X[-1]).ravel()

ranked = []
for c, ts in zip(candidates, text_scores):
    rs = rule_score(c)
    raw = (0.40 * float(ts) + 0.45 * rs + 0.15) * behavior(c) * anomaly_penalty(c)
    ranked.append((c.get('candidate_id'), raw, c, ts, rs))
ranked.sort(key=lambda x: (-x[1], str(x[0])))

max_s = max(x[1] for x in ranked) or 1
min_s = min(x[1] for x in ranked)
den = max(max_s - min_s, 1e-9)
rows = []
for i, (cid, raw, c, ts, rs) in enumerate(ranked[:100], 1):
    score = round((raw - min_s) / den, 4)
    skills = c.get('skills', [])
    skills_txt = ', '.join(skills[:3]) if isinstance(skills, list) else str(skills)
    reasoning = f"{c.get('headline','Candidate')} with {c.get('years_experience', c.get('experience_years','?'))} yrs; matched retrieval/ranking JD through {skills_txt}. Rule fit={rs:.2f}, text relevance={float(ts):.2f}."
    rows.append({'candidate_id': cid, 'rank': i, 'score': score, 'reasoning': reasoning})

out = pd.DataFrame(rows)
out.to_csv('ranked_output.csv', index=False)
out


,candidate_id,rank,score,reasoning
0,CAND_DEMO_001,1,1.0000,Senior Machine Learning Engineer with 7.2 yrs;...
1,CAND_DEMO_005,2,0.7377,Search Engineer with 6.8 yrs; matched retrieva...
2,CAND_DEMO_003,3,0.6521,NLP Engineer with 6.4 yrs; matched retrieval/r...
3,CAND_DEMO_002,4,0.2334,Backend Engineer with 5.5 yrs; matched retriev...
4,CAND_DEMO_004,5,0.0000,AI Research Intern with 0.7 yrs; matched retri...


## 4. Download ranked CSV


In [ ]:
try:
    from google.colab import files
    files.download('ranked_output.csv')
except Exception:
    print('Saved to ranked_output.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>